In [283]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [284]:
# data = pd.read_stata(r"Z:\harmonized\ECU\ENEMDU\data_arm\ECU_1990m11_BID.dta") # para bases de stata
data = pd.read_stata(r"C:\Users\ronal\OneDrive\Documentos\GitHub\salario-minimo-y-pobreza-Ecuador\BASES EN STATA\ECU_2003m12.dta") # para bases de stata # para bases de stata

## Revisar los datos

- rn - región natural
- estrato - estrato
- cuidad - ciudad
- zona - zona
- sector - sector
- vivienda - vivienda
- hogar - hogar
- persona - persona
- numpers - número de personas
- edad - edad
- ingpat - Ingresos como patrono o cuenta propia
- ingasg - Ingresos como asalariado de gobierno
- ingepv - Ingresos asalariado empresa privada
- ingdom - Ingresos como empleada doméstica
- ingalq - Ingresos por alquileres, rentas o interese
- ingjub - Ingresos por jubilación o pensión
- ingotr - por otros ingresos
- fexp - factor de expansión
- ingrl - ingresos

El valor de 'ingasg' e 'ingepv' son el ingreso laboral monetario vamos a incluir 'ingdom' para darle un scope mayor, no hay datos sobre ingreso laboral no monetario, las otras variables son ingreso no laboral monetario y no monetario e ingrl es un ingreso total

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, se puede usar estas variables y el ingreso laboral asumiendo que cuando estaba trabajando tenía ese ingreso para intentar aproximar el salario mensual y de ahí el salario trimestral, esto solo funciona así ya que no tenemos una variable que explicite el mes, en encuestas que tengan el mes o trimestre explícito esto no sería igual.

In [285]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82317 entries, 0 to 82316
Columns: 273 entries, rgnal to bien6
dtypes: category(170), float32(1), float64(73), int16(4), int32(1), int8(20), object(4)
memory usage: 64.6+ MB


Filtramos solo las columnas de interés para alivar el peso en la memoria

In [286]:

data.columns

Index(['rgnal', 'rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 'panelm',
       'vivienda', 'hogar',
       ...
       'ii6', 'ii3', 'ii5', 'bien3', 'bien1', 'bien2', 'bien5', 'bien7',
       'bien4', 'bien6'],
      dtype='object', length=273)

In [287]:
data.columns = data.columns.str.strip()  # Elimina espacios al principio y al final
data = data[['rn','ciudad','zona','sector', 'vivienda', 'hogar', 'persona', 'numpers', 'edad','pe63', 'fexp','ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado en el sector público, asalariado en el sector privado o ingresos por empleo doméstico

In [288]:
data['ingr'] = data[['pe63']]

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados en un mes, ahora las variables categoricas por mes tienen diferentes leyendas
- trabajando
- buscando trabajo
- sin buscar trabajo

In [289]:
data['ingr_ene'] = data.apply(lambda x: x['ingr'] if x['ene'] == 'trabajando' else None, axis=1)
data['ingr_feb'] = data.apply(lambda x: x['ingr'] if x['feb'] == 'trabajando' else None, axis=1)
data['ingr_mar'] = data.apply(lambda x: x['ingr'] if x['mar'] == 'trabajando' else None, axis=1)
data['ingr_abr'] = data.apply(lambda x: x['ingr'] if x['abr'] == 'trabajando' else None, axis=1)
data['ingr_may'] = data.apply(lambda x: x['ingr'] if x['may'] == 'trabajando' else None, axis=1)
data['ingr_jun'] = data.apply(lambda x: x['ingr'] if x['jun'] == 'trabajando' else None, axis=1)
data['ingr_jul'] = data.apply(lambda x: x['ingr'] if x['jul'] == 'trabajando' else None, axis=1)
data['ingr_ago'] = data.apply(lambda x: x['ingr'] if x['ago'] == 'trabajando' else None, axis=1)
data['ingr_sep'] = data.apply(lambda x: x['ingr'] if x['sep'] == 'trabajando' else None, axis=1)
data['ingr_oct'] = data.apply(lambda x: x['ingr'] if x['oct'] == 'trabajando' else None, axis=1)
data['ingr_nov'] = data.apply(lambda x: x['ingr'] if x['nov'] == 'trabajando' else None, axis=1)
data['ingr_dic'] = data.apply(lambda x: x['ingr'] if x['dic'] == 'trabajando' else None, axis=1)

Ingreso mensual promedio en el trimeste

In [290]:
data['ingr_t1'] = (data['ingr_ene'] + data['ingr_feb'] + data['ingr_mar'])/3
data['ingr_t2'] = (data['ingr_abr'] + data['ingr_may'] + data['ingr_jun'])/3
data['ingr_t3'] = (data['ingr_jul'] + data['ingr_ago'] + data['ingr_sep'])/3
data['ingr_t4'] = (data['ingr_oct'] + data['ingr_nov'] + data['ingr_dic'])/3

## Corregimos los valores de ser necesario

In [291]:
# Llenar valores perdidos con un dato (0)
data['ingr_t1'] = data['ingr_t1'].fillna(0)
data['ingr_t2'] = data['ingr_t2'].fillna(0)
data['ingr_t3'] = data['ingr_t3'].fillna(0)
data['ingr_t4'] = data['ingr_t4'].fillna(0)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [292]:
# Carga base de datos con ipc
data_externa = pd.read_excel(r"C:\Users\ronal\OneDrive\Documentos\GitHub\salario-minimo-y-pobreza-Ecuador\data_externa.xlsx", sheet_name='datos')




# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2003]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc y tipo de cambio

In [293]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_base.iterrows()
     }

tipo_cambio_dict = dict(zip(datos_actual['trimestre'], datos_actual['tipo de cambio']))

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [294]:
# Corregimos los códigos para usarlos cómo texto
data['ciudad'] = data['ciudad'].apply(str)
data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data['ciudad_2'] = data['ciudad'].apply(lambda x: x[:2])

Diccionario ciudades disponibles

In [295]:
parroquia_dict = {
    '01': 'Cuenca',
    '09': 'Guayaquil',
    '17': 'Quito'
}

data['ciudad_asignada'] = data['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))

Elegimos el trimestre según las variables del INEC donde sea igual a 'trabajando'

In [296]:
def assgna_trimestre(fila):
    if 'trabajando' in [fila['ene'], fila['feb'], fila['mar']]:
        return 1
    elif 'trabajando' in [fila['abr'], fila['may'], fila['jun']]:
        return 2
    elif 'trabajando' in [fila['jul'], fila['ago'], fila['sep']]:
        return 3
    elif 'trabajando' in [fila['oct'], fila['nov'], fila['dic']]:
        return 4
    else:
        return None
    
data['trimestre'] = data.apply(assgna_trimestre, axis=1)

### Asignamos el ipc y tipo de cambio correspondiente según trimestre y ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = \frac{ingr_{sucres}^{i}}{tipo-de-cambio^{i}}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

In [297]:
# Función que asigna valores correspondientes
def asigna_ipc(fila):
    return ipc_dict.get(fila['trimestre'], {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila):
    return ipc_base_dict.get(fila['trimestre'], {}).get(fila['ciudad_asignada'], None)

In [298]:
data['ipc'] = data.apply(asigna_ipc, axis=1)
data['ipc_base'] = data.apply(asigna_ipc_base, axis=1)
data['tipo_cambio'] = data['trimestre'].map(tipo_cambio_dict)

In [299]:
# Calculamos el deflactor
data['def'] = (data['ipc_base'] / data['ipc'])

In [300]:
data['ingr_t1_r'] = (data['ingr_t1'] / data['tipo_cambio']) * data['def']
data['ingr_t2_r'] = (data['ingr_t2'] / data['tipo_cambio']) * data['def']
data['ingr_t3_r'] = (data['ingr_t3'] / data['tipo_cambio']) * data['def']
data['ingr_t4_r'] = (data['ingr_t4'] / data['tipo_cambio']) * data['def']

In [301]:
data[['ingr_t1', 'ingr_t2', 'ingr_t3', 'ingr_t4', 'ipc', 'ipc_base', 'def', 'tipo_cambio', 'ingr_t1_r', 'ingr_t2_r', 'ingr_t3_r', 'ingr_t4_r']]

,ingr_t1,ingr_t2,ingr_t3,ingr_t4,ipc,ipc_base,def,tipo_cambio,ingr_t1_r,ingr_t2_r,ingr_t3_r,ingr_t4_r
0,150.0,0.0,0.0,150.0,66.054661,97.862086,1.481532,NaN,NaN,NaN,NaN,NaN
1,0.0,0.0,0.0,0.0,66.054661,99.405769,1.504902,NaN,NaN,NaN,NaN,NaN
2,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
82312,0.0,0.0,0.0,0.0,65.025974,98.083034,1.508367,NaN,NaN,NaN,NaN,NaN
82313,0.0,0.0,0.0,0.0,65.025974,98.083034,1.508367,NaN,NaN,NaN,NaN,NaN
82314,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82315,0.0,0.0,0.0,0.0,65.025974,98.083034,1.508367,NaN,NaN,NaN,NaN,NaN


## Calculo ingreso de los hogares

In [302]:
columnas_idef = ['rn', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

4998

In [303]:
data[['rn', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar', 'idef_hogar', 'persona', 'numpers']]

,rn,ciudad,zona,sector,vivienda,hogar,idef_hogar,persona,numpers
0,sierra,010150,1,5,1,1,sierra0101501511,01,4
1,sierra,010150,1,5,1,1,sierra0101501511,02,4
2,sierra,010150,1,5,1,1,sierra0101501511,04,4
3,sierra,010150,1,5,1,1,sierra0101501511,03,4
4,sierra,010150,1,5,2,1,sierra0101501521,05,5
...,...,...,...,...,...,...,...,...,...
82312,4,900351,999,45,3,1,49003519994531,05,7
82313,4,900351,999,45,3,1,49003519994531,06,7
82314,4,900351,999,45,3,1,49003519994531,02,7
82315,4,900351,999,45,3,1,49003519994531,01,7


In [304]:
data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform('sum')
data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform('sum')
data['ingr_t3_h'] = data.groupby('idef_hogar')['ingr_t3_r'].transform('sum')
data['ingr_t4_h'] = data.groupby('idef_hogar')['ingr_t4_r'].transform('sum')

In [305]:
data[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h', 'ingr_t4_h']]

,ingr_t1_h,ingr_t2_h,ingr_t3_h,ingr_t4_h
0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0
...,...,...,...,...
82312,0.0,0.0,0.0,0.0
82313,0.0,0.0,0.0,0.0
82314,0.0,0.0,0.0,0.0
82315,0.0,0.0,0.0,0.0


In [306]:
print("Ingreso medio de un hogar t4: ", data['ingr_t4_h'].mean())
print("Mediana del ingreso de un hogar t4: ", data['ingr_t4_h'].median())

Ingreso medio de un hogar t4:  0.0
Mediana del ingreso de un hogar t4:  0.0


## Sacamos edades negativas y mayores a 100 años

In [307]:
len(data)

82317

En este caso en la variable edad tenemos números y el texto 'menos de un año' así que primero transformamos todas las filas que digan 'menos de un año' a 0

In [308]:
data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)

In [309]:
data = data.loc[(data['edad'] >= 0) & (data['edad'] < 100)]
len(data)

82317

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [310]:
k = 0.4
s = 0.9

In [311]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

In [312]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

In [313]:
data['ingr_t_t1'] = data['ingr_t1_h'] / data['escala']
data['ingr_t_t2'] = data['ingr_t2_h'] / data['escala']
data['ingr_t_t3'] = data['ingr_t3_h'] / data['escala']
data['ingr_t_t4'] = data['ingr_t4_h'] / data['escala']

In [314]:
data[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3', 'ingr_t_t4']]

,ingr_t_t1,ingr_t_t2,ingr_t_t3,ingr_t_t4
0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0
...,...,...,...,...
82312,0.0,0.0,0.0,0.0
82313,0.0,0.0,0.0,0.0
82314,0.0,0.0,0.0,0.0
82315,0.0,0.0,0.0,0.0


In [315]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  0.0
Mediana del ingreso individual descontando cargas familiares t4:  0.0


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [316]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))

In [317]:
# Asigna el umbral por trimestre si hay umbral por región modificar
data['umbral'] = data['trimestre'].map(umbral_dict)

data['umbral'] = data.groupby('idef_hogar')['umbral'].transform('mean')

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

In [318]:
datos_final = pd.DataFrame(index=['t1', 't2', 't3', 't4'], columns=['fgt0', 'fgt1', 'fgt2', 'a25', 'a50', 'a75', 'ingreso_promedio'])

In [319]:
data['persona_fexp'] = 1 * data['fexp']

In [320]:
for t in [1, 2, 3, 4]:
    condicion = data['trimestre'] == t
    col_ingr = f'ingr_t_t{t}'
    col_pobres = f'pobres_t{t}'

    # una columna que identifica a quienes están por debajo de la línea de pobreza por trimestre
    data.loc[condicion, col_pobres] = (
        (data.loc[condicion, col_ingr] - data.loc[condicion, 'umbral']) < 0
    ).astype(int)

In [321]:
print("pobreza t1: ", (data['pobres_t1'] * data['fexp']).sum()/data.loc[data['trimestre'] == 1]['persona_fexp'].sum())
print("pobreza t2: ", (data['pobres_t2'] * data['fexp']).sum()/data.loc[data['trimestre'] == 2]['persona_fexp'].sum())
print("pobreza t3: ", (data['pobres_t3'] * data['fexp']).sum()/data.loc[data['trimestre'] == 3]['persona_fexp'].sum())
print("pobreza t4: ", (data['pobres_t4'] * data['fexp']).sum()/data.loc[data['trimestre'] == 4]['persona_fexp'].sum())

pobreza t1:  0.9999999145640401
pobreza t2:  0.9999999491219216
pobreza t3:  1.0000000926134642
pobreza t4:  0.9999999487130816


In [322]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data['trimestre'] == t].copy()

    # Calculamos una columna de pobres
    df_temp['pobres'] = (df_temp[f'ingr_t_t{t}'] - umbral_dict[t]) < 0

    # Ratio de pobres sobre el total
    ratio = (umbral_dict[t] - df_temp[f'ingr_t_t{t}']) / umbral_dict[t]

    # Calculamos el índice para alpha 0, 1 y 2 solo donde 'pobres' == True.
    for i in range(3):
        col = f'fgt{i}'
        df_temp[col] = np.where(df_temp['pobres'], ratio**i, 0)

    # Cálculo del índice ponderado: se usa el factor de expansión como peso
    peso_total = df_temp['fexp'].sum()
    fgt0 = (df_temp['fgt0'] * df_temp['fexp']).sum() / peso_total
    fgt1 = (df_temp['fgt1'] * df_temp['fexp']).sum() / peso_total
    fgt2 = (df_temp['fgt2'] * df_temp['fexp']).sum() / peso_total
    
    # Guardamos los resultados
    datos_final.loc[f't{t}', 'fgt0'] = fgt0
    datos_final.loc[f't{t}', 'fgt1'] = fgt1
    datos_final.loc[f't{t}', 'fgt2'] = fgt2

In [323]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,1.0,1.0,1.0,NaN,NaN,NaN,NaN
t2,1.0,1.0,1.0,NaN,NaN,NaN,NaN
t3,1.0,1.0,1.0,NaN,NaN,NaN,NaN
t4,1.0,1.0,1.0,NaN,NaN,NaN,NaN


## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [324]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data['trimestre'] == t].copy()

    # Suma total de los factores de expansión para el trimestre
    peso_total = df_temp['fexp'].sum()
    
    # Ingreso promedio ponderado
    mu = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total

    # Calculamos el índice A para epsilon 0.25, 0.5 y 0.75 utilizando los pesos
    indices = {}
    for i in [0.25, 0.5, 0.75]:
        A_i = ((df_temp[f'ingr_t_t{t}']**(1-i) * df_temp['fexp']).sum() / peso_total)**(1/(1-i))
        indices[i] = A_i

    # Ratio de pobreza con el índice total (aplicando la fórmula)
    a25 = 1 - 1/mu * indices[0.25]
    a50 = 1 - 1/mu * indices[0.5]
    a75 = 1 - 1/mu * indices[0.75]

    # Guardamos los resultados
    datos_final.loc[f't{t}', 'a25'] = a25
    datos_final.loc[f't{t}', 'a50'] = a50
    datos_final.loc[f't{t}', 'a75'] = a75


C:\Users\ronal\AppData\Local\Temp\ipykernel_25240\1535860004.py:18: RuntimeWarning: divide by zero encountered in scalar divide
  a25 = 1 - 1/mu * indices[0.25]
C:\Users\ronal\AppData\Local\Temp\ipykernel_25240\1535860004.py:18: RuntimeWarning: invalid value encountered in scalar multiply
  a25 = 1 - 1/mu * indices[0.25]
C:\Users\ronal\AppData\Local\Temp\ipykernel_25240\1535860004.py:19: RuntimeWarning: divide by zero encountered in scalar divide
  a50 = 1 - 1/mu * indices[0.5]
C:\Users\ronal\AppData\Local\Temp\ipykernel_25240\1535860004.py:19: RuntimeWarning: invalid value encountered in scalar multiply
  a50 = 1 - 1/mu * indices[0.5]
C:\Users\ronal\AppData\Local\Temp\ipykernel_25240\1535860004.py:20: RuntimeWarning: divide by zero encountered in scalar divide
  a75 = 1 - 1/mu * indices[0.75]
C:\Users\ronal\AppData\Local\Temp\ipykernel_25240\1535860004.py:20: RuntimeWarning: invalid value encountered in scalar multiply
  a75 = 1 - 1/mu * indices[0.75]


In [325]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,1.0,1.0,1.0,NaN,NaN,NaN,NaN
t2,1.0,1.0,1.0,NaN,NaN,NaN,NaN
t3,1.0,1.0,1.0,NaN,NaN,NaN,NaN
t4,1.0,1.0,1.0,NaN,NaN,NaN,NaN


Guardamos el ingreso promedio

In [326]:
for t in [1, 2, 3, 4]:
    df_temp = data.loc[data['trimestre'] == t].copy()
    
    # Calcula la suma total de los factores de expansión
    peso_total = df_temp['fexp'].sum()
    
    # Calcula el ingreso promedio ponderado
    media_ponderada = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total
    
    datos_final.loc[f't{t}', 'ingreso_promedio'] = media_ponderada


In [327]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,1.0,1.0,1.0,NaN,NaN,NaN,0.0
t2,1.0,1.0,1.0,NaN,NaN,NaN,0.0
t3,1.0,1.0,1.0,NaN,NaN,NaN,0.0
t4,1.0,1.0,1.0,NaN,NaN,NaN,0.0


In [328]:
datos_final.to_csv('datos_final2003.csv')